### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [4]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


c:\Users\crist\Desktop\CEIA\4 - PNL\procesamiento_lenguaje_natural\.venv\Scripts\python.exe: No module named pip


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [6]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [7]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [8]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [9]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [10]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [11]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [12]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [13]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [14]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [15]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [16]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [17]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [18]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [19]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [20]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [21]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [22]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [23]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [24]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [25]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [26]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


# Punto 1

In [28]:
longitudes = np.array([len(doc) for doc in newsgroups_train.data])
print(f'Documentos de train totalmente vacíos: {(longitudes == 0).sum()}')
print(f'Documentos de train con 300 caracteres o menos: {(longitudes <= 300).sum()} de {len(longitudes)}')

rng = np.random.default_rng(42)
candidatos = np.where(longitudes > 300)[0]
docs_elegidos = rng.choice(candidatos, size=5, replace=False)

print(f'\nDocumentos elegidos al azar: {docs_elegidos}')
for i in docs_elegidos:
    print(f'  doc {i:5d} | {longitudes[i]:5d} caracteres | clase: {newsgroups_train.target_names[y_train[i]]}')

Documentos de train totalmente vacíos: 218
Documentos de train con 300 caracteres o menos: 3694 de 11314

Documentos elegidos al azar: [8784 4942 7434 1008 4874]
  doc  8784 |   328 caracteres | clase: comp.sys.ibm.pc.hardware
  doc  4942 |   571 caracteres | clase: comp.windows.x
  doc  7434 |  3964 caracteres | clase: rec.sport.hockey
  doc  1008 |   397 caracteres | clase: comp.sys.ibm.pc.hardware
  doc  4874 |   447 caracteres | clase: talk.politics.mideast


In [31]:
def palabras_en_comun(i, j, n=5):
    """Palabras que más aportan a la similaridad entre los documentos i y j."""
    producto = X_train[i].multiply(X_train[j]).toarray()[0]
    top = np.argsort(producto)[::-1][:n]
    return [idx2word[k] for k in top if producto[k] > 0]


def mostrar_similares(idx, n=5, chars=350):
    """Muestra un documento y sus n vecinos más similares, con su clase."""
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mas_similares = np.argsort(cossim)[::-1][1:n + 1]   # [0] es el propio documento (sim = 1)

    print('=' * 100)
    print(f'DOCUMENTO {idx}  |  clase: {newsgroups_train.target_names[y_train[idx]]}')
    print('-' * 100)
    texto = newsgroups_train.data[idx].strip()
    print(texto[:chars] + ('...' if len(texto) > chars else ''))
    print('-' * 100)
    print(f'{"sim":>6}  {"doc":>5}  {"misma":>5}  {"clase":<26}  palabras que explican la similaridad')

    for j in mas_similares:
        misma = 'SI' if y_train[j] == y_train[idx] else 'no'
        clase = newsgroups_train.target_names[y_train[j]]
        print(f'{cossim[j]:6.3f}  {j:5d}  {misma:>5}  {clase:<26}  {", ".join(palabras_en_comun(idx, j))}')
    print()


for i in docs_elegidos:
    mostrar_similares(i)

DOCUMENTO 8784  |  clase: comp.sys.ibm.pc.hardware
----------------------------------------------------------------------------------------------------
Please reply via e-mail since this is job related: 

I have a Colorado Jumbo back-up system at one of my places of 
employment and it has eaten two tapes by winding the tape off the spool.

Is there an easy fix or is the tape drive fried? Does it simply need 
cleaning?

Any and all comments will be appreciated!

Stephen Husak
----------------------------------------------------------------------------------------------------
   sim    doc  misma  clase                       palabras que explican la similaridad
 0.334  11244     SI  comp.sys.ibm.pc.hardware    tape, tapes, the, drive, is
 0.239   1585     SI  comp.sys.ibm.pc.hardware    tape, tapes, colorado, jumbo, drive
 0.221    745     SI  comp.sys.ibm.pc.hardware    tape, the, colorado, is, system
 0.219   1546     SI  comp.sys.ibm.pc.hardware    tape, the, drive, is, jumbo
 0.216  

In [32]:
def palabras_en_comun(i, j, n=5):
    """Palabras que más aportan a la similaridad entre los documentos i y j."""
    producto = X_train[i].multiply(X_train[j]).toarray()[0]
    top = np.argsort(producto)[::-1][:n]
    return [idx2word[k] for k in top if producto[k] > 0]


def resumir(texto, chars):
    """Deja el texto en una sola línea y lo recorta."""
    texto = ' '.join(texto.split())
    return texto[:chars] + ('...' if len(texto) > chars else '')


def mostrar_similares(idx, n=5, chars=400, chars_vecino=230):
    """Muestra un documento y sus n vecinos más similares: clase, texto y palabras en común."""
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mas_similares = np.argsort(cossim)[::-1][1:n + 1]   # [0] es el propio documento (sim = 1)

    print('=' * 100)
    print(f'DOCUMENTO {idx}  |  clase: {newsgroups_train.target_names[y_train[idx]]}')
    print('=' * 100)
    print(resumir(newsgroups_train.data[idx], chars))

    for k, j in enumerate(mas_similares, 1):
        misma = 'MISMA CLASE' if y_train[j] == y_train[idx] else 'otra clase'
        print('-' * 100)
        print(f'{k}. doc {j}  |  sim = {cossim[j]:.3f}  |  {newsgroups_train.target_names[y_train[j]]} ({misma})')
        print(f'   explican: {", ".join(palabras_en_comun(idx, j))}')
        print(f'   {resumir(newsgroups_train.data[j], chars_vecino)}')
    print()


for i in docs_elegidos:
    mostrar_similares(i)

DOCUMENTO 8784  |  clase: comp.sys.ibm.pc.hardware
Please reply via e-mail since this is job related: I have a Colorado Jumbo back-up system at one of my places of employment and it has eaten two tapes by winding the tape off the spool. Is there an easy fix or is the tape drive fried? Does it simply need cleaning? Any and all comments will be appreciated! Stephen Husak
----------------------------------------------------------------------------------------------------
1. doc 11244  |  sim = 0.334  |  comp.sys.ibm.pc.hardware (MISMA CLASE)
   explican: tape, tapes, the, drive, is
   Does it do it to all tapes? Were the two tapes that it unwound of the same type from the same batch? The reason I ask is that I bought some generic tapes that did this and found that the tape markers were not fully punched out and...
----------------------------------------------------------------------------------------------------
2. doc 1585  |  sim = 0.239  |  comp.sys.ibm.pc.hardware (MISMA CLASE)
   ex

### Interpretación 1

De los 25 vecinos recuperados (5 por cada documento), 15 quedaron en la misma clase que su documento de origen. Igual el promedio no dice mucho, porque los cinco casos se comportaron muy distinto.

**Documento 8784 (comp.sys.ibm.pc.hardware).** Una consulta sobre una unidad de cinta Colorado Jumbo que arruinó dos cintas. Los 5 vecinos son de la misma clase y todos hablan de unidades de cinta, con `tape`, `colorado`, `jumbo` y `drive` como palabras que explican la similaridad. Es el mejor caso posible, porque son términos muy específicos que casi no aparecen en el resto del corpus y por eso el IDF les da mucho peso.

**Documento 4942 (comp.windows.x).** Un problema de teclas con Emacs y xmodmap. Los dos primeros vecinos aciertan y comparten `emacs`, `shift` y `xmodmap`, pero del tercero en adelante aparecen discursos de la Casa Blanca y un post sobre criptografía que no tienen nada que ver. Lo que los explica son `and`, `to`, `the`. Se nota además en el valor de similaridad, que cae de 0.378 a 0.210 y 0.196, así que abajo de 0.20 la similaridad ya es ruido.

**Documento 7434 (rec.sport.hockey).** Un pronóstico de playoffs de la NHL. Los tres primeros vecinos son del mismo tema y comparten `montreal`, `gilmour` y `team`, pero el cuarto y el quinto son sobre Azerbaiyán y ateísmo, con similaridades altas (0.346 y 0.337) explicadas solo por stopwords. Creo que es porque este es el documento más largo de los cinco (3964 caracteres) y al tener tantas palabras termina solapándose un poco con cualquier texto largo.

**Documento 1008 (comp.sys.ibm.pc.hardware).** Un problema de color de 16 y 24 bits en un sistema Gateway. Tres vecinos son de la misma clase, pero los otros dos, aunque figuran como error, en realidad hablan del mismo tema (placas de video ATI y SCSI) y solo están etiquetados como comp.os.ms-windows.misc y comp.sys.mac.hardware. Acá la similaridad funcionó bien y el problema es que las clases comp.* se superponen entre sí.

**Documento 4874 (talk.politics.mideast).** Una acusación agresiva escrita en segunda persona. Es el peor caso, porque solo el primer vecino se sostiene por contenido y el resto (letras de Black Sabbath, una discusión bíblica) aparece explicado por `you`, `your` y `are`. Lo que comparten no es el tema sino la forma de escribir, con el mismo tono de confrontación.

**Conclusión.** La similaridad coseno sobre TF-IDF anda muy bien cuando el documento tiene vocabulario específico y poco frecuente, y falla cuando lo que comparten los textos son stopwords, la longitud o el registro de escritura.


# Punto 2

In [33]:
def clasificar_por_similaridad(Xte, Xtr, y_tr, batch=500):
    """A cada documento de test le asigna la clase del documento de train más parecido."""
    pred = np.empty(Xte.shape[0], dtype=int)
    max_sim = np.empty(Xte.shape[0])

    for ini in range(0, Xte.shape[0], batch):
        fin = min(ini + batch, Xte.shape[0])
        sims = cosine_similarity(Xte[ini:fin], Xtr)   # (batch, 11314)
        vecino = sims.argmax(axis=1)
        pred[ini:fin] = y_tr[vecino]
        max_sim[ini:fin] = sims[np.arange(fin - ini), vecino]

    return pred, max_sim


y_pred_zs, max_sim = clasificar_por_similaridad(X_test, X_train, y_train)

print(f'F1 macro (zero-shot, vecino más cercano): {f1_score(y_test, y_pred_zs, average="macro"):.4f}')
print(f'F1 macro (Naive Bayes de la plantilla):   {f1_score(y_test, y_pred, average="macro"):.4f}')
print()
print(f'Similaridad con el vecino elegido -> mediana: {np.median(max_sim):.3f} | media: {max_sim.mean():.3f}')
print(f'Documentos de test que no comparten NINGUNA palabra con train: {(max_sim == 0).sum()}')
print(f'Documentos de test decididos con similaridad < 0.2: {(max_sim < 0.2).sum()} de {len(max_sim)}')


F1 macro (zero-shot, vecino más cercano): 0.5050
F1 macro (Naive Bayes de la plantilla):   0.5854

Similaridad con el vecino elegido -> mediana: 0.281 | media: 0.308
Documentos de test que no comparten NINGUNA palabra con train: 224
Documentos de test decididos con similaridad < 0.2: 1128 de 7532


In [34]:
f1_por_clase = f1_score(y_test, y_pred_zs, average=None)
orden = np.argsort(f1_por_clase)

print('Clases donde MEJOR funciona el vecino más cercano:')
for i in orden[::-1][:5]:
    print(f'  {f1_por_clase[i]:.3f}  {newsgroups_train.target_names[i]}')

print('\nClases donde PEOR funciona:')
for i in orden[:5]:
    print(f'  {f1_por_clase[i]:.3f}  {newsgroups_train.target_names[i]}')

# Los 224 documentos de test sin ninguna palabra en común con train
vacios = max_sim == 0
print(f'\nDe los {vacios.sum()} documentos sin palabras en común, acertó: {(y_pred_zs[vacios] == y_test[vacios]).sum()}')
print(f'A todos les asignó la clase: {newsgroups_train.target_names[y_pred_zs[vacios][0]]}')

# Cuánto pesa la falta de evidencia
confiable = max_sim >= 0.2
print(f'\nAccuracy sobre las {confiable.sum()} decisiones con sim >= 0.2: {(y_pred_zs[confiable] == y_test[confiable]).mean():.3f}')
print(f'Accuracy sobre las {(~confiable).sum()} decisiones con sim <  0.2: {(y_pred_zs[~confiable] == y_test[~confiable]).mean():.3f}')


Clases donde MEJOR funciona el vecino más cercano:
  0.735  rec.sport.hockey
  0.642  comp.windows.x
  0.586  rec.sport.baseball
  0.570  sci.crypt
  0.569  rec.motorcycles

Clases donde PEOR funciona:
  0.277  talk.religion.misc
  0.307  talk.politics.misc
  0.406  sci.electronics
  0.425  alt.atheism
  0.453  talk.politics.mideast

De los 224 documentos sin palabras en común, acertó: 23
A todos les asignó la clase: rec.autos

Accuracy sobre las 6404 decisiones con sim >= 0.2: 0.548
Accuracy sobre las 1128 decisiones con sim <  0.2: 0.285


### Interpretación del Punto 2

El clasificador no tiene entrenamiento. El modelo es directamente la matriz `X_train` con sus etiquetas, y para cada documento de test busco el documento de train más parecido por similaridad coseno y le copio la clase. Dio **F1 macro de 0.505**, casi 8 puntos por debajo del Naïve Bayes de la plantilla (0.585).

**Por qué queda abajo.** Naïve Bayes usa todos los documentos de cada clase para estimar qué palabras la caracterizan, mientras que acá la decisión se toma mirando un único documento. Si ese documento es raro, corto o comparte solo palabras genéricas, la predicción se va al fondo. Es un método de varianza muy alta.

**El problema de fondo es la falta de evidencia.** La mediana de la similaridad con el vecino elegido es 0.281, y en el Punto 1 ya había visto que abajo de 0.20 lo que se comparte son stopwords. Separando por ese umbral queda clarísimo, porque las 6404 decisiones tomadas con similaridad mayor o igual a 0.2 aciertan un 54.8%, y las 1128 tomadas con menos de 0.2 aciertan apenas un 28.5%. El clasificador igual está obligado a elegir algo, aunque no tenga con qué.

**Un caso extremo.** Hay 224 documentos de test que quedaron vacíos después del `remove=('headers','footers','quotes')` y no comparten ninguna palabra con train. Su similaridad es cero contra todo, así que `argmax` devuelve el índice 0 por defecto y los 224 reciben la clase del primer documento de train, que es `rec.autos`. Acertó 23 de puro azar. No son predicciones sino un artefacto de implementación, y me pareció importante detectarlo porque de otra forma quedan escondidos dentro del F1.

**Qué clases funcionan y cuáles no.** Las mejores son `rec.sport.hockey` (0.735), `comp.windows.x` (0.642) y `rec.sport.baseball` (0.586), todas clases con jerga propia y muy poco ambigua, como nombres de equipos y jugadores o funciones de X Window. Las peores son `talk.religion.misc` (0.277), `talk.politics.misc` (0.307) y `alt.atheism` (0.425). Las tres son categorías tipo cajón de sastre, definidas más por el tono de discusión que por un vocabulario propio, y se pisan entre ellas y con `soc.religion.christian` y `talk.politics.mideast`. Es exactamente lo que había visto en el documento 4874 del Punto 1, donde los vecinos se parecían por escribir en segunda persona y no por el tema.


# Punto 3

In [35]:
import pandas as pd

vectorizadores = {
    'tfidf (por defecto)':            TfidfVectorizer(),
    'tfidf + stopwords':              TfidfVectorizer(stop_words='english'),
    'tfidf + sw + min_df=2':          TfidfVectorizer(stop_words='english', min_df=2),
    'tfidf + sw + min_df=2 + sublin': TfidfVectorizer(stop_words='english', min_df=2, sublinear_tf=True),
    'tfidf + sw + max_df=0.5':        TfidfVectorizer(stop_words='english', max_df=0.5),
    'conteos + stopwords':            CountVectorizer(stop_words='english'),
}
modelos = {'MultinomialNB': MultinomialNB, 'ComplementNB': ComplementNB}
alphas = [1.0, 0.5, 0.1, 0.05, 0.01]

filas = []
for nombre_vec, vec in vectorizadores.items():
    Xtr = vec.fit_transform(newsgroups_train.data)   # el vectorizador se ajusta una sola vez
    Xte = vec.transform(newsgroups_test.data)
    for nombre_mod, Modelo in modelos.items():
        for a in alphas:
            pred = Modelo(alpha=a).fit(Xtr, y_train).predict(Xte)
            filas.append({'vectorizador': nombre_vec, 'modelo': nombre_mod, 'alpha': a,
                          'vocabulario': Xtr.shape[1],
                          'f1_macro': f1_score(y_test, pred, average='macro')})

resultados = pd.DataFrame(filas).sort_values('f1_macro', ascending=False).reset_index(drop=True)

print(f'Baseline de la plantilla (MultinomialNB alpha=1 sobre tfidf por defecto): {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'Combinaciones probadas: {len(resultados)}\n')
print('TOP 10:')
print(resultados.head(10).to_string(index=False))


Baseline de la plantilla (MultinomialNB alpha=1 sobre tfidf por defecto): 0.5854
Combinaciones probadas: 60

TOP 10:
                  vectorizador       modelo  alpha  vocabulario  f1_macro
             tfidf + stopwords ComplementNB    0.5       101322  0.697805
       tfidf + sw + max_df=0.5 ComplementNB    0.5       101322  0.697805
         tfidf + sw + min_df=2 ComplementNB    0.5        39115  0.697363
           tfidf (por defecto) ComplementNB    0.5       101631  0.696116
           tfidf (por defecto) ComplementNB    0.1       101631  0.695365
tfidf + sw + min_df=2 + sublin ComplementNB    0.5        39115  0.695108
         tfidf + sw + min_df=2 ComplementNB    1.0        39115  0.694292
             tfidf + stopwords ComplementNB    1.0       101322  0.693611
       tfidf + sw + max_df=0.5 ComplementNB    1.0       101322  0.693611
           tfidf (por defecto) ComplementNB    1.0       101631  0.692953


In [36]:
tabla = resultados.pivot_table(index='vectorizador', columns=['modelo', 'alpha'], values='f1_macro')
tabla = tabla.reindex(list(vectorizadores.keys()))
print('F1 macro para cada combinación (filas = vectorizador, columnas = modelo y alpha)\n')
print(tabla.round(3).to_string())


F1 macro para cada combinación (filas = vectorizador, columnas = modelo y alpha)

modelo                         ComplementNB                             MultinomialNB                            
alpha                                  0.01   0.05   0.10   0.50   1.00          0.01   0.05   0.10   0.50   1.00
vectorizador                                                                                                     
tfidf (por defecto)                   0.669  0.686  0.695  0.696  0.693         0.683  0.670  0.656  0.615  0.585
tfidf + stopwords                     0.665  0.684  0.692  0.698  0.694         0.684  0.680  0.673  0.658  0.647
tfidf + sw + min_df=2                 0.673  0.682  0.689  0.697  0.694         0.680  0.683  0.680  0.663  0.651
tfidf + sw + min_df=2 + sublin        0.673  0.684  0.688  0.695  0.692         0.673  0.679  0.678  0.657  0.644
tfidf + sw + max_df=0.5               0.665  0.684  0.692  0.698  0.694         0.684  0.680  0.673  0.658  0.647
conteo

In [37]:
mejor = resultados.iloc[0]
vec_mejor = vectorizadores[mejor['vectorizador']]
Xtr = vec_mejor.fit_transform(newsgroups_train.data)
Xte = vec_mejor.transform(newsgroups_test.data)
y_pred_best = modelos[mejor['modelo']](alpha=mejor['alpha']).fit(Xtr, y_train).predict(Xte)

print(f'Mejor combinación: {mejor["modelo"]} (alpha={mejor["alpha"]}) sobre "{mejor["vectorizador"]}"')
print(f'F1 macro = {f1_score(y_test, y_pred_best, average="macro"):.4f}  (baseline de la plantilla: {f1_score(y_test, y_pred, average="macro"):.4f})\n')

comparacion = pd.DataFrame({
    'zero-shot (P2)': f1_score(y_test, y_pred_zs, average=None),
    'baseline (plantilla)': f1_score(y_test, y_pred, average=None),
    'mejor modelo (P3)': f1_score(y_test, y_pred_best, average=None),
    'docs en train': np.bincount(y_train),
}, index=newsgroups_train.target_names)
comparacion['mejora vs baseline'] = comparacion['mejor modelo (P3)'] - comparacion['baseline (plantilla)']

print(comparacion.sort_values('mejor modelo (P3)', ascending=False).round(3).to_string())


Mejor combinación: ComplementNB (alpha=0.5) sobre "tfidf + stopwords"
F1 macro = 0.6978  (baseline de la plantilla: 0.5854)

                          zero-shot (P2)  baseline (plantilla)  mejor modelo (P3)  docs en train  mejora vs baseline
rec.sport.hockey                   0.735                 0.714              0.897            600               0.183
rec.sport.baseball                 0.586                 0.801              0.877            597               0.076
talk.politics.mideast              0.453                 0.769              0.814            564               0.045
rec.motorcycles                    0.569                 0.735              0.806            598               0.071
sci.med                            0.561                 0.730              0.806            594               0.076
comp.windows.x                     0.642                 0.781              0.800            593               0.019
sci.space                          0.566                

### Interpretación del Punto 3

Probé 60 combinaciones cruzando seis vectorizadores, los dos modelos de Naïve Bayes y cinco valores de `alpha`, sin tocar `ngram_range`. La mejor fue **ComplementNB con alpha=0.5 sobre TF-IDF con `stop_words='english'`, F1 macro de 0.698** contra el 0.585 de la plantilla.

**El modelo pesa más que el vectorizador.** Las 10 mejores combinaciones son todas ComplementNB. Estima, para cada clase, qué palabras caracterizan a todo lo que NO es esa clase, así que trabaja siempre con muestras grandes y eso lo hace mucho más estable. Además tolera mejor que le demos valores TF-IDF decimales en lugar de los conteos enteros que asume el modelo multinomial.

**El `alpha` es lo que más cambia a MultinomialNB**, que sube de 0.585 a 0.683 solo bajándolo de 1 a 0.01. Creo que es un problema de escala: los valores TF-IDF son mucho menores que 1 porque cada documento se normaliza a norma 1, así que sumarle 1 a cada uno de los 101.631 términos del vocabulario es agregar muchísima evidencia inventada. ComplementNB en cambio se mueve apenas entre 0.665 y 0.698 en todo el rango.

**Sacar stopwords y bajar el alpha arreglan lo mismo.** En MultinomialNB con alpha=1 las stopwords suben el F1 de 0.585 a 0.647, pero con alpha=0.01 no aportan nada. Son las palabras con más masa de conteo, o sea las más amplificadas por el suavizado excesivo.

**Otros dos resultados.** Los conteos crudos nunca superan 0.645, así que el IDF aporta de verdad. Y `min_df=2` consigue el mismo F1 con 39.115 palabras en lugar de 101.631, lo que muestra que casi dos tercios del vocabulario son palabras de un solo documento.

**De dónde salen los 11 puntos.** Las clases que más mejoran son las que el baseline tenía casi muertas: `talk.religion.misc` de 0.008 a 0.221, `talk.politics.misc` de 0.147 a 0.517 y `alt.atheism` de 0.127 a 0.366. Son las que menos documentos de train tienen. Con alpha=1 el suavizado aplastaba las diferencias y el modelo elegía siempre las clases grandes, y como el F1 macro pondera igual a todas, rescatarlas es lo que mueve la métrica.

**Lo que no se arregla.** Las peores siguen siendo `talk.religion.misc`, `alt.atheism` y `talk.politics.misc`, las mismas que fallaban en el Punto 2. Son categorías cajón de sastre que se solapan con `soc.religion.christian` y `talk.politics.guns`, y eso no se resuelve con hiperparámetros.

**Aclaración metodológica.** Elegí `alpha` mirando el mismo test que después reporto, así que el 0.698 está algo optimista. Lo hice así porque la consigna pide maximizar el F1 en test, y el riesgo es bajo porque ComplementNB varía apenas medio punto en todo el rango.



# Punto 4

In [40]:
tfidf_palabras = TfidfVectorizer(stop_words='english', min_df=5, sublinear_tf=True)
X_train_p = tfidf_palabras.fit_transform(newsgroups_train.data)

X_terminos = X_train_p.T.tocsr()          # matriz término-documento
vocab_p = tfidf_palabras.vocabulary_
idx2word_p = {v: k for k, v in vocab_p.items()}
print(f'Matriz documento-término: {X_train_p.shape}  ->  matriz término-documento: {X_terminos.shape}\n')


def palabras_similares(palabra, n=5):
    if palabra not in vocab_p:
        print(f'"{palabra}" no está en el vocabulario')
        return
    i = vocab_p[palabra]
    cossim = cosine_similarity(X_terminos[i], X_terminos)[0]
    similares = [j for j in np.argsort(cossim)[::-1] if j != i][:n]

    vecinos = ', '.join(f'{idx2word_p[j]} ({cossim[j]:.2f})' for j in similares)
    print(f'{palabra:>10} (en {X_terminos[i].nnz:4d} docs) -> {vecinos}')


# Palabras elegidas a mano, una por área temática distinta
for p in ['car', 'windows', 'god', 'israel', 'doctor']:
    palabras_similares(p)


Matriz documento-término: (11314, 17797)  ->  matriz término-documento: (17797, 11314)

       car (en  396 docs) -> cars (0.20), dealer (0.19), civic (0.17), owner (0.15), engine (0.14)
   windows (en  575 docs) -> dos (0.32), ms (0.24), files (0.20), file (0.20), microsoft (0.20)
       god (en  579 docs) -> jesus (0.30), faith (0.27), christ (0.27), bible (0.26), lord (0.24)
    israel (en  184 docs) -> israeli (0.45), arab (0.38), arabs (0.34), lebanon (0.33), syrian (0.32)
    doctor (en  100 docs) -> ultrasound (0.21), patient (0.20), urgent (0.20), clinic (0.17), upset (0.17)


In [41]:
# Mismas palabras pero sobre el vocabulario COMPLETO de la plantilla (sin stopwords ni min_df)
X_terminos_full = X_train.T.tocsr()


def similares_full(palabra, n=5):
    i = tfidfvect.vocabulary_[palabra]
    cossim = cosine_similarity(X_terminos_full[i], X_terminos_full)[0]
    similares = [j for j in np.argsort(cossim)[::-1] if j != i][:n]

    vecinos = ', '.join(f'{idx2word[j]} ({cossim[j]:.2f})' for j in similares)
    print(f'{palabra:>10} (en {X_terminos_full[i].nnz:4d} docs) -> {vecinos}')


print('Vocabulario COMPLETO (101.631 términos, sin filtrar):')
for p in ['car', 'doctor', 'goalie', 'thyroid']:
    similares_full(p)

print('\nVocabulario FILTRADO (17.797 términos, stopwords + min_df=5):')
for p in ['car', 'doctor', 'goalie', 'thyroid']:
    palabras_similares(p)

# Relación entre frecuencia de la palabra y calidad de sus vecinos
frecuencias = np.diff(X_terminos.indptr)      # en cuántos documentos aparece cada término
print(f'\nEn el vocabulario filtrado, la mediana de documentos por palabra es {int(np.median(frecuencias))}')
print(f'Palabras que aparecen en menos de 10 documentos: {(frecuencias < 10).sum()} de {len(frecuencias)}')


Vocabulario COMPLETO (101.631 términos, sin filtrar):
       car (en  396 docs) -> cars (0.18), criterium (0.18), civic (0.17), owner (0.17), dealer (0.17)
    doctor (en  100 docs) -> receptionist (0.44), clinic (0.33), misbehavior (0.30), urgent (0.29), angering (0.29)
    goalie (en   28 docs) -> hardening (0.49), cheeks (0.49), essensa (0.46), plaster (0.40), forehead (0.39)
   thyroid (en    1 docs) -> menstrual (1.00), laryngitis (1.00), internist (1.00), dermatologist (1.00), endocrinologist (0.75)

Vocabulario FILTRADO (17.797 términos, stopwords + min_df=5):
       car (en  396 docs) -> cars (0.20), dealer (0.19), civic (0.17), owner (0.15), engine (0.14)
    doctor (en  100 docs) -> ultrasound (0.21), patient (0.20), urgent (0.20), clinic (0.17), upset (0.17)
    goalie (en   28 docs) -> essensa (0.43), shutouts (0.28), mask (0.28), nominations (0.28), ahl (0.25)
"thyroid" no está en el vocabulario

En el vocabulario filtrado, la mediana de documentos por palabra es 12
Palabr

### Interpretación del Punto 4

Al transponer, cada fila pasa a ser una palabra y su vector dice en qué documentos aparece. Dos palabras son similares si aparecen en los mismos documentos. Es la hipótesis distributiva de la Clase 2, pero con el documento entero como contexto en lugar de una ventana de pocas palabras.

**Los resultados son muy buenos para lo simple que es el método.** `car` trae `cars`, `dealer`, `civic` y `engine`. `windows` trae `dos`, `ms`, `microsoft` y `files`. `god` trae `jesus`, `faith`, `christ` y `bible`. `israel` trae `israeli`, `arab`, `lebanon` y `syrian`, con las similaridades más altas de las cinco (0.45 contra 0.14 de `car`), porque el tema está muy concentrado en una sola clase mientras que hablar de autos aparece desperdigado en varios grupos.

**Pero no son sinónimos, son palabras que comparten tema.** `windows` y `dos` no significan lo mismo, aparecen juntas. Lo que se captura es co-ocurrencia temática, no significado. Es la diferencia con los embeddings tipo Word2Vec, que al usar ventanas chicas capturan relaciones más finas.

**El caso más flojo es `doctor`**, que además de `ultrasound`, `patient` y `clinic` trae `urgent` y `upset`. Es también la que aparece en menos documentos (100), y esa relación no es casualidad.

**La segunda celda lo confirma.** Ordenando palabras de más a menos frecuente, la degradación es gradual. `goalie`, con 28 documentos y sin filtro, devuelve `hardening`, `cheeks`, `plaster` y `forehead`, que son palabras de un único post sobre la máscara de un arquero, y ese post alcanza para dominar todo su vector. El caso límite es `thyroid`, que aparece en un solo documento y llega a similaridad 1.00 con `menstrual`, `laryngitis` e `internist`. No son sinónimos, comparten el único documento en el que existen, así que sus vectores son idénticos salvo por la escala.

**Un detalle de implementación que salió de ahí.** Al principio excluía la palabra consultada tomando las posiciones `[1:n+1]`, asumiendo que siempre queda primera con similaridad 1. Con `thyroid` falló, porque había cinco términos empatados en 1.00 y el orden entre empates de `argsort` es arbitrario. Lo corregí excluyéndola por índice en lugar de por posición.

**Conclusión.** El método anda bien con palabras frecuentes y con carga temática, y se rompe con palabras raras, donde el vector refleja un par de documentos puntuales en vez de un patrón de uso. Incluso con `min_df=5` quedan 7356 palabras de 17797 que aparecen en menos de 10 documentos, así que casi la mitad del vocabulario sigue siendo poco confiable. Por eso la consigna pide elegirlas a mano, porque al azar habrían salido casi todas del tipo de `thyroid`.



# Conclusiones generales

**La representación importa más que el truco.** Los cuatro puntos usan la misma matriz TF-IDF y la misma similaridad coseno, y todo lo que funcionó o falló se explica por cómo esa representación reparte el peso entre las palabras. Funciona cuando el texto tiene vocabulario específico y poco frecuente, y se degrada cuando lo que comparten los documentos es longitud, stopwords o registro de escritura.

**Mirar qué palabras explican una similaridad fue la herramienta más útil de todo el trabajo.** Sirvió en el Punto 1 para distinguir una similaridad temática de una por stopwords, y en el Punto 4 para descubrir que los vecinos de una palabra rara venían todos de un único post.

**Los tres clasificadores quedaron ordenados así:** vecino más cercano 0.505, Naïve Bayes de la plantilla 0.585 y ComplementNB ajustado 0.698. El vecino más cercano pierde porque decide con un solo documento y la mitad de sus decisiones se toman con similaridad menor a 0.28. Naïve Bayes gana porque promedia la evidencia de toda la clase.

**Lo que movió la aguja en el Punto 3 no fue el vectorizador sino el modelo y el suavizado.** Cambiar a ComplementNB y calibrar `alpha` a la escala del TF-IDF valió 11 puntos, mientras que las stopwords, `min_df` y `max_df` aportaron menos de un punto cada uno. Sí sirvieron para otra cosa: `min_df=2` da el mismo F1 con un tercio del vocabulario.

**Las clases difíciles son las mismas en los tres modelos.** `talk.religion.misc`, `alt.atheism` y `talk.politics.misc` fallan en el zero-shot, en el baseline y en el mejor modelo. Son categorías cajón de sastre que se solapan con `soc.religion.christian` y `talk.politics.guns`, así que el límite no está en el modelo sino en que las etiquetas son ambiguas hasta para un humano.

**Una limitación del trabajo.** Los hiperparámetros los elegí mirando el F1 del mismo conjunto de test que después reporto, así que el 0.698 es algo optimista. Lo correcto sería seleccionarlos por validación cruzada sobre train y usar test una sola vez.
